Mình sẽ rà nhanh tài liệu kiến trúc ev-node và các entrypoint trong block để đề xuất đúng các điểm “nối” giúp cosmos chain của bạn hoàn thiện hơn theo hướng production.

Read skill [ev-node-explainer](file:///Users/nguyennganha/code/ev-node/.claude/skills/ev-node-explainer/SKILL.md?vscodeLinkType%3Dskill)

Read [](file:///Users/nguyennganha/code/ev-node/block/public.go)

Read [](file:///Users/nguyennganha/code/ev-node/block/components.go#1-1), lines 1 to 280

Read [](file:///Users/nguyennganha/code/ev-node/block/components.go#280-280), lines 280 to 381

Mình đã có khung component-level; giờ mình đọc thêm `submitter/syncer` để đưa đề xuất nối cụ thể theo luồng DA, catchup và forced inclusion (không nói chung chung).

Read [](file:///Users/nguyennganha/code/ev-node/block/internal/submitting/submitter.go#1-1), lines 1 to 280

Read [](file:///Users/nguyennganha/code/ev-node/block/internal/syncing/syncer.go#1-1), lines 1 to 320

Read [](file:///Users/nguyennganha/code/ev-node/block/internal/syncing/da_follower.go#1-1), lines 1 to 280

Searched for regex `NewAggregatorWithCatchupComponents|newAggregatorComponents|NewSyncComponents|NewDAClient\(` (`**/node/**|apps/**|scripts/**|block/**|pkg/**`), no results

Searched codebase for "block.NewSyncComponents NewAggregatorWithCatchupComponents NewDAClient used in node setup", 11 results

Read [](file:///Users/nguyennganha/code/ev-node/node/failover.go#1-1), lines 1 to 260

Read [](file:///Users/nguyennganha/code/ev-node/pkg/rpc/server/server.go#240-240), lines 240 to 360

Read [](file:///Users/nguyennganha/code/ev-node/pkg/rpc/server/da_visualization.go#1-1), lines 1 to 260

Searched for regex `tx|block|search|explorer|/da|height|ws` (`**/pkg/rpc/server/**`), 20 results

Có, và thực ra chain Cosmos WASM của bạn đã nối đúng “xương sống” ev-node rồi; phần cần làm tiếp là nối sâu hơn để thành hệ hoàn chỉnh.

**Điểm đã nối tốt**
- Mode aggregator/sync + catchup đã dùng đúng block components qua failover.go (gọi NewAggregatorWithCatchupComponents và NewSyncComponents).
- Vòng đời block chuẩn ev-node có đủ Executor/Reaper/Submitter/Syncer trong components.go.
- DA client public + namespace handling đã có trong public.go.

**Các điểm nên nối thêm để chain “hoàn thiện hơn”**
- Nối Forced Inclusion end-to-end (chống censorship) từ syncer.go và forced_inclusion_retriever.go vào policy vận hành Cosmos (grace window, alert khi vi phạm).
- Nối DA follower/catchup observability (head lag, retry, timeout) từ da_follower.go vào dashboard/alert.
- Nối Submitter metrics + trạng thái retry/oversize từ submitter.go vào telemetry tập trung.
- Nối RPC explorer backend trực tiếp từ server.go + da_visualization.go để làm web scan (block/tx/DA submissions/blob details) thay vì chỉ script CLI.
- Nối persistence/indexer riêng (Postgres/Elastic) đọc từ RPC + p2p stores để query nhanh, filter sâu, thống kê.

**Roadmap triển khai nhanh (ưu tiên)**
- P1: Explorer API ổn định (block/tx/search + DA endpoints hiện có).
- P2: Indexer DB + reindex job theo height.
- P3: Forced inclusion monitoring + SLO cảnh báo.
- P4: Hardening production (key management, backup, chaos/recovery test).

Nếu bạn muốn, mình có thể chốt ngay bản thiết kế “web scan MVP” bám đúng endpoint hiện có trong repo để team bắt tay code luôn.